In [1]:
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer
import pandas as pd
from datasets import load_dataset

W0221 19:01:17.268000 6460 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
model_id = r'google/gemma-3-270m-it'
dataset_path = r'../../data/train_data/messages_intr.csv'

In [3]:
model = AutoModelForCausalLM.from_pretrained(model_id, device_map='auto')
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.chat_template = "{% for message in messages %}{{'<start_of_turn>' + message['role'] + '\n' + message['content'] + '<end_of_turn>' + '\n'}}{% endfor %}"
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [4]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.1,
    bias='none',
    task_type='CAUSAL_LM'
)

In [5]:
data = load_dataset("csv", data_files=dataset_path, split="train")

In [6]:
system_prompt = 'Выделяй суть не более чем в 5 словах'

In [7]:
def format_chat_template(sample):
    return {
        'messages': [
            {'role': 'user', 'content': sample['instruction']},
            {'role': 'assistant', 'content': sample['response']}
        ]
    }

In [8]:
dataset = data.map(format_chat_template, remove_columns=['instruction', 'response'])

In [9]:
dataset

Dataset({
    features: ['messages'],
    num_rows: 3
})

In [10]:
from trl import SFTTrainer, SFTConfig

In [13]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=SFTConfig(
        output_dir=r'../../models/gemma-lora-result',
        per_device_train_batch_size=4,
        learning_rate=2e-4,
        num_train_epochs=3,
        logging_steps=10,
        fp16=True,
        bf16=False
    ),
    peft_config=lora_config,
    processing_class=tokenizer
)

Tokenizing train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


In [14]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 1}.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Denis\_netrc.
wandb: Currently logged in as: salnikovdenis44 (salnikovdenis44-omstu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


C:\Users\Denis\miniconda3\envs\myenv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


TrainOutput(global_step=3, training_loss=5.480406443277995, metrics={'train_runtime': 23.3178, 'train_samples_per_second': 0.386, 'train_steps_per_second': 0.129, 'total_flos': 690015456000.0, 'train_loss': 5.480406443277995, 'entropy': 2.854021151860555, 'num_tokens': 939.0, 'mean_token_accuracy': 0.3655913968880971, 'epoch': 3.0})

In [15]:
from peft import PeftModel

adapter_path = r'../../models/gemma-lora-result/checkpoint-3'

#base_model = AutoModelForCausalLM.from_pretrained(model_id, device_map='auto')
peft_model = PeftModel.from_pretrained(model, adapter_path)

# Склеиваем адаптер с базовой моделью
merged_model = peft_model.merge_and_unload()

# Сохраняем итоговую модель
merged_model.save_pretrained(r'../../models/gemma-merged')
tokenizer.save_pretrained(r'../../models/gemma-merged')

C:\Users\Denis\miniconda3\envs\myenv\lib\site-packages\peft\tuners\tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


('../../models/gemma-merged\\tokenizer_config.json',
 '../../models/gemma-merged\\special_tokens_map.json',
 '../../models/gemma-merged\\chat_template.jinja',
 '../../models/gemma-merged\\tokenizer.model',
 '../../models/gemma-merged\\added_tokens.json',
 '../../models/gemma-merged\\tokenizer.json')